# Hermes Agent — Colab Heartbeat Launcher

**How to use:** Click **Runtime → Run all** (Ctrl+F9). State auto-restores from Drive. When this VM dies, you'll get a Discord notification with a restore link.

In [ ]:
#@title Step 1: Restore state from Drive + start heartbeat
import shutil, json, os, subprocess, sys, time, threading, atexit
from pathlib import Path
from datetime import datetime, timezone

# === CONFIG (pre-filled) ===
BACKUP = Path('/content/drive/MyDrive/hermes-colab/hermes-state')
HOME = Path('/root/.hermes')
GIST_ID = '9739987481e693ba7cea5c53597356b0'
GH_TOKEN = 'YOUR_GITHUB_TOKEN'  # <-- Replace with your GitHub token
COLAB_URL = 'https://colab.research.google.com/drive/1yfzxR6YpvzGLa2G9GOmEDekH1ne75LtQ'
# ============================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

# Restore state
skip = {'auth.lock','gateway.lock','gateway.pid','gateway.sock','gateway-starts.log'}
if BACKUP.exists():
    HOME.mkdir(exist_ok=True)
    for item in BACKUP.iterdir():
        if item.name in skip: continue
        dest = HOME / item.name
        if item.is_dir():
            if dest.exists(): shutil.rmtree(dest)
            shutil.copytree(item, dest)
        else:
            shutil.copy2(item, dest)
    print('State restored from Drive')
else:
    print('No backup found — starting fresh')

# Shutdown hook: sync back
def sync_to_drive():
    for item in HOME.iterdir():
        if item.name in skip: continue
        dest = BACKUP / item.name
        if item.is_dir():
            if dest.exists(): shutil.rmtree(dest)
            shutil.copytree(item, dest)
        else:
            shutil.copy2(item, dest)
    print('State synced to Drive')
atexit.register(sync_to_drive)

# Heartbeat to GitHub Gist
import urllib.request
def write_heartbeat():
    ts = datetime.now(timezone.utc).isoformat()
    payload = json.dumps({"files": {"heartbeat.json": {"content": json.dumps({"status":"alive","timestamp":ts,"colab_url":COLAB_URL})}}}).encode()
    req = urllib.request.Request(f'https://api.github.com/gists/{GIST_ID}', data=payload, method='PATCH')
    req.add_header('Authorization', f'token {GH_TOKEN}')
    req.add_header('Content-Type', 'application/json')
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            if r.status == 200: print(f'Heartbeat: {ts}')
    except Exception as e:
        print(f'Heartbeat failed: {e}')

def heartbeat_loop():
    while True:
        write_heartbeat()
        time.sleep(300)

threading.Thread(target=heartbeat_loop, daemon=True).start()
print('Heartbeat started (every 5 min)')

In [ ]:
#@title Step 2: Start Hermes Gateway
import subprocess
proc = subprocess.Popen(
    ['hermes', 'gateway', 'run'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in proc.stdout:
    print(line, end='')